### Lab: Multi-Tool Agent Build

<small>Build a bounded research assistant that searches, summarizes, and saves a final note.  
Learners should implement the three tool bodies, the ReAct loop, error recovery, and a decision log.  
Use a local task boundary only: no unrestricted web browsing, no shell access, and no unapproved writes.</small>

<small>**Task scenario**  
A research assistant receives a question, searches a small approved knowledge source, summarizes the result, and saves the answer.  
The task is intentionally bounded so the agent can focus on planning, tool use, and recovery instead of open-ended browsing.  

**Rubric requirement**  
A valid solution must include:  
- explicit loop limits,  
- permission boundaries,  
- a decision log,  
- and a high-stakes approval gate for any write action.</small>

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path

class Tool(ABC):
    name = ""
    description = ""
    schema = {}
    REQUIRES_APPROVAL = False  # subclasses set True for high-stakes write actions

    @abstractmethod
    def run(self, **_kwargs):
        _ = _kwargs
        raise NotImplementedError


# Global intermittent failure state for demonstrations
RATE_LIMIT_STATE = {"counter": 0, "limit_every": 3}

class SearchTool(Tool):
    name = "search"
    description = "Search an approved local knowledge source."
    schema = {
        "name": "search",
        "description": "Search a bounded knowledge source and return ranked matches.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "minimum": 1, "maximum": 5},
            },
            "required": ["query"],
        },
    }

    def __init__(self, corpus_path):
        self.corpus_path = Path(corpus_path)

    def run(self, **kwargs):
        query = kwargs.get("query", "")
        top_k = kwargs.get("top_k", 3)
        q_lower = query.lower()

        # Deterministic simulated failures triggered by keywords
        if "timeout" in q_lower:
            return {"status": "error", "error_type": "timeout", "message": "simulated timeout"}
        if "rate_limit" in q_lower or "rate-limit" in q_lower:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated rate limit (keyword)"}
        if "malformed" in q_lower:
            return "THIS IS MALFORMED OUTPUT"

        # Intermittent rate-limit: flip on every Nth call to simulate transient throttling
        RATE_LIMIT_STATE["counter"] += 1
        if RATE_LIMIT_STATE["counter"] % RATE_LIMIT_STATE["limit_every"] == 0:
            return {"status": "error", "error_type": "rate_limit", "message": "simulated intermittent rate limit"}

        matches = []
        for p in self.corpus_path.glob("*.txt"):
            text = p.read_text(encoding="utf-8")
            score = text.lower().count(query.lower())
            if score > 0:
                matches.append({"file": p.name, "score": score, "text": text})
        matches.sort(key=lambda x: x["score"], reverse=True)
        return {"status": "success", "results": matches[:top_k]}


class SummarizeTool(Tool):
    name = "summarize"
    description = "Summarize retrieved content into a short answer."
    schema = {
        "name": "summarize",
        "description": "Summarize search results into a concise response.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "style": {"type": "string", "enum": ["short", "bullet"]},
            },
            "required": ["text"],
        },
    }

    def run(self, **kwargs):
        text = kwargs.get("text", "")
        style = kwargs.get("style", "short")
        if style == "bullet":
            sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 20]
            summary = "\n".join(f"• {s}." for s in sentences[:4])
        else:
            summary = (text.split(".")[0].strip() + ".") if "." in text else text[:200]
        return {"status": "success", "summary": summary}


class SaveTool(Tool):
    """Write the final research note to disk.

    Flagged REQUIRES_APPROVAL = True because saving is a write action
    that persists beyond the agent session. The loop must gate this tool
    behind a human confirmation step before calling run().
    """
    name = "save"
    description = (
        "Save a research summary to a local note file. "
        "HIGH-STAKES write action — requires human approval before execution."
    )
    REQUIRES_APPROVAL = True
    schema = {
        "name": "save",
        "description": (
            "Save a research summary to the output folder. "
            "Requires explicit human approval."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "Target filename (no path)"},
                "content": {"type": "string", "description": "Text content to save"},
            },
            "required": ["filename", "content"],
        },
    }

    def run(self, **kwargs):
        filename = kwargs.get("filename", "research_note.txt")
        content = kwargs.get("content", "")
        out_dir = Path("output")
        out_dir.mkdir(exist_ok=True)
        # Restrict to output/ only — strip any path components to prevent traversal
        safe_name = Path(filename).name
        out_path = out_dir / safe_name
        out_path.write_text(content, encoding="utf-8")
        return {"status": "success", "saved_to": str(out_path), "bytes": len(content)}


# instantiate with local corpus
CORPUS_DIR = Path("Track2/Labs/data/LabC2.3_corpus").resolve()

TOOL_LIBRARY = {
    "search":    SearchTool(CORPUS_DIR),
    "summarize": SummarizeTool(),
    "save":      SaveTool(),
}

print("Tool library ready:", list(TOOL_LIBRARY.keys()))
for name, tool in TOOL_LIBRARY.items():
    approval_flag = "⚠ REQUIRES_APPROVAL" if tool.REQUIRES_APPROVAL else "✓ auto"
    print(f"  {name:12s} — {approval_flag} — {tool.description[:60]}")

<small>**ReAct loop requirements**  
The agent should alternate between reasoning, choosing one tool, observing the result, and updating state.  
It must stop when the task is complete, when the iteration limit is reached, or when repeated failures make progress unsafe.  

**Error recovery requirement**  
Do not use a bare try/except as the only answer.  
The agent should classify failures, retry with a change in behavior, re-plan when the output shape is wrong, and stop safely when recovery is not possible.</small>

In [ ]:
import time

MAX_ITERATIONS = 8
MAX_RETRIES    = 2
# Save is in the allowed set but gated behind approval — not excluded, just controlled
ALLOWED_TOOLS  = {"search", "summarize", "save"}

decision_log = []
RATE_LIMIT_STATE["counter"] = 0  # reset for clean demo


def classify_tool_result(result):
    if not isinstance(result, dict):
        return "unexpected_output"
    if result.get("status") == "error":
        return result.get("error_type", "unknown_error")
    if result.get("status") in {"success", "skipped"}:
        return "ok"
    return "unexpected_output"


def recovery_plan(error_type, retry_count):
    if error_type in {"timeout", "rate_limit"} and retry_count < MAX_RETRIES:
        return {"decision": "retry", "retry_count": retry_count + 1, "wait": 2 ** retry_count}
    if error_type == "unexpected_output":
        return {"decision": "replan", "retry_count": retry_count}
    return {"decision": "stop", "retry_count": retry_count}


def request_approval(tool_name, tool_input, simulated_answer=True):
    """
    In production use input() for real interactive approval.
    Here we use simulated_answer=True to allow the demo to run automatically.
    Set simulated_answer=False to demonstrate a denial.
    """
    print(f"\n[APPROVAL REQUIRED] tool={tool_name}")
    print(f"  Proposed input: {tool_input}")
    print(f"  Simulated decision: {'APPROVED' if simulated_answer else 'DENIED'}")
    return simulated_answer


def run_agent_step(state):
    """Deterministic planner: search → summarize → save (with approval)."""
    step = state.get("step", 1)
    query = state.get("query", "agent")

    if step == 1:
        state["step"] = 2
        return (
            f"Step 1 — Search corpus for '{query}'",
            {"tool": "search", "input": {"query": query, "top_k": 3}},
        )

    if step == 2:
        state["step"] = 3
        last = state.get("observations", [])[-1] if state.get("observations") else {}
        if not isinstance(last, dict) or last.get("status") != "success":
            # malformed or failed search — retry search
            state["step"] = 1
            return ("Step 2 — Retry search after bad result", {"tool": "search", "input": {"query": query, "top_k": 3}})
        texts = "\n".join(r.get("text", "") for r in last.get("results", []))
        return (
            "Step 2 — Summarize retrieved documents",
            {"tool": "summarize", "input": {"text": texts, "style": "bullet"}},
        )

    if step == 3:
        state["step"] = 4
        summary = ""
        for obs in reversed(state.get("observations", [])):
            if isinstance(obs, dict) and obs.get("status") == "success" and "summary" in obs:
                summary = obs["summary"]
                break
        return (
            "Step 3 — Save summary (requires human approval)",
            {"tool": "save", "input": {"filename": f"{query.replace(' ', '_')}_note.txt", "content": summary}},
        )

    return ("NoOp — goal already complete", {"tool": "search", "input": {"query": "noop", "top_k": 1}})


# ── Run the agent ────────────────────────────────────────────────────────────
state = {"goal_complete": False, "observations": [], "query": "agent safety"}
retry_count = 0
decision_log.clear()

for iteration in range(MAX_ITERATIONS):
    thought, action = run_agent_step(state)
    tool_name  = action["tool"]
    tool_input = action["input"]

    # ① Permission boundary
    if tool_name not in ALLOWED_TOOLS:
        decision_log.append({"iteration": iteration, "tool": tool_name, "status": "permission_denied"})
        print(f"[BLOCKED] {tool_name} is not in the allowed tool set.")
        break

    # ② Approval gate for high-stakes tools
    if TOOL_LIBRARY[tool_name].REQUIRES_APPROVAL:
        approved = request_approval(tool_name, tool_input, simulated_answer=True)
        if not approved:
            decision_log.append({"iteration": iteration, "tool": tool_name, "status": "approval_denied"})
            state["goal_complete"] = True          # treat denial as graceful stop
            state["final_status"]  = "save_declined_by_user"
            break
        decision_log.append({"iteration": iteration, "tool": tool_name, "status": "approval_granted"})

    # ③ Execute tool
    result = TOOL_LIBRARY[tool_name].run(**tool_input)
    decision_log.append({
        "iteration": iteration,
        "thought":   thought,
        "tool":      tool_name,
        "input":     tool_input,
        "result":    result,
    })

    # ④ Error classification & recovery
    result_state = classify_tool_result(result)
    if result_state != "ok":
        recovery = recovery_plan(result_state, retry_count)
        decision_log.append({"iteration": iteration, "recovery": recovery})
        if recovery["decision"] == "retry":
            retry_count = recovery["retry_count"]
            time.sleep(0)          # replace with time.sleep(recovery["wait"]) in production
            continue
        if recovery["decision"] == "replan":
            state["observations"].append({"repair_needed": result_state})
            continue
        state["final_status"] = "stopped_safely"
        break

    retry_count = 0
    state["observations"].append(result)

    # ⑤ Goal check
    if tool_name == "save" and result.get("status") == "success":
        state["goal_complete"] = True
        state["final_status"]  = "completed"
        break
else:
    state.setdefault("final_status", "stopped_by_iteration_limit")

# ── Decision log summary ─────────────────────────────────────────────────────
print(f"\nFinal status : {state.get('final_status', 'unknown')}")
print(f"Goal complete: {state['goal_complete']}")
print(f"\nDecision log ({len(decision_log)} entries):")
for entry in decision_log:
    i     = entry.get("iteration", "–")
    tool  = entry.get("tool", "–")
    st    = entry.get("status") or entry.get("result", {}).get("status", "–") if isinstance(entry.get("result"), dict) else entry.get("status", "–")
    rec   = entry.get("recovery", {}).get("decision", "") if entry.get("recovery") else ""
    thought_snippet = entry.get("thought", "")[:60]
    print(f"  [{i}] {tool:12s}  status={st:20s}  recovery={rec:8s}  thought={thought_snippet}")

<small>Interpretation: The agent searched the local corpus and matched `doc_001.txt` and `doc_003.txt`. The summarizer extracted the first sentence as a concise answer. The agent then attempted to save the note, but the save step is gated by human approval (not yet granted), demonstrating the safety boundary.</small>

#### LLM-driven ReAct Loop

<small>The deterministic loop above hard-codes the step sequence.  
A real agent uses the LLM to decide which tool to call next, based on what it has observed so far.  
The cell below implements a full Claude-powered ReAct loop with:

- tool selection driven by the model  
- approval gate for `save`  
- permission boundary enforced before every tool call  
- structured decision log at each step  

Swap `query` for any domain question to reuse the same loop across finance, education, healthcare, and e-commerce scenarios.</small>

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Build the tool schema list from the tool library
LLM_TOOL_SCHEMAS = [
    tool.schema for tool in TOOL_LIBRARY.values()
    if tool.schema
]

LLM_ALLOWED_TOOLS = {"search", "summarize", "save"}

SYSTEM_PROMPT = """You are a bounded research assistant.
Your only knowledge sources are the tools available to you.
Follow this process strictly:
1. Use the search tool to find relevant information from the approved knowledge base.
2. Use the summarize tool to condense the findings into a clear answer.
3. Use the save tool to persist the final answer (this requires human approval).
Always reason about what you found before choosing the next tool.
Stop as soon as the answer is saved or you cannot make further progress."""


def run_llm_react_agent(query, max_iterations=8, auto_approve_save=True):
    """
    Full LLM-driven ReAct loop using Claude tool use.

    Parameters
    ----------
    query            : The research question to answer.
    max_iterations   : Hard cap on reasoning cycles.
    auto_approve_save: Set False to simulate a user denying the save action.
    """
    messages = [{"role": "user", "content": query}]
    llm_decision_log = []

    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=LLM_TOOL_SCHEMAS,
            messages=messages,
        )

        # Capture any reasoning text the model produced
        reasoning = " | ".join(
            b.text for b in response.content if hasattr(b, "text") and b.text
        )
        log_entry = {
            "iteration":   iteration,
            "stop_reason": response.stop_reason,
            "thought":     reasoning,
        }

        # ── Model finished without needing more tools ──
        if response.stop_reason == "end_turn":
            llm_decision_log.append(log_entry)
            final_text = reasoning
            print(f"\n[DONE] Agent finished after {iteration + 1} iteration(s).")
            print(f"Answer: {final_text}")
            return {"answer": final_text, "decision_log": llm_decision_log}

        # ── Process tool calls ──
        tool_results = []
        for block in response.content:
            if not hasattr(block, "type") or block.type != "tool_use":
                continue

            tool_name  = block.name
            tool_input = block.input
            log_entry["tool"]  = tool_name
            log_entry["input"] = tool_input

            # ① Permission boundary
            if tool_name not in LLM_ALLOWED_TOOLS:
                tool_result = {"status": "error", "message": f"Tool '{tool_name}' is not permitted."}
                log_entry["permission"] = "denied"
                print(f"[BLOCKED] {tool_name} is outside the allowed tool set.")

            # ② Approval gate for high-stakes tools
            elif TOOL_LIBRARY.get(tool_name, Tool()).REQUIRES_APPROVAL:
                print(f"\n[APPROVAL REQUIRED] tool={tool_name}")
                print(f"  Filename : {tool_input.get('filename', '?')}")
                print(f"  Content  : {str(tool_input.get('content', ''))[:120]}...")
                granted = auto_approve_save
                print(f"  Decision : {'APPROVED' if granted else 'DENIED'}")
                log_entry["approval"] = "granted" if granted else "denied"
                if granted:
                    tool_result = TOOL_LIBRARY[tool_name].run(**tool_input)
                else:
                    tool_result = {"status": "skipped", "message": "Save declined by user."}

            # ③ Normal execution
            else:
                tool_result = TOOL_LIBRARY[tool_name].run(**tool_input)

            log_entry["result"] = tool_result
            print(f"  [{iteration}] {tool_name} → {str(tool_result)[:100]}")

            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     json.dumps(tool_result),
            })

        llm_decision_log.append(log_entry)

        # Feed tool results back to the model
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user",      "content": tool_results})

    return {
        "answer":       "Max iterations reached without completion.",
        "decision_log": llm_decision_log,
    }


# ── Domain scenario selector ─────────────────────────────────────────────────
# Change `query` to explore different domains against the corpus.

scenarios = {
    "general":     "What is an AI agent and how does the ReAct pattern work?",
    "finance":     "What safety boundaries should an AI agent have when handling financial transactions?",
    "education":   "Summarise how an AI agent can help a student plan their course sequence.",
    "healthcare":  "What approval and logging requirements apply to agents that schedule medical appointments?",
    "ecommerce":   "How should an agent handle tool failures when tracking a customer order?",
}

# Pick one scenario to run
query = scenarios["finance"]
print(f"Running agent for domain scenario: finance")
print(f"Query: {query}\n")

result = run_llm_react_agent(query, max_iterations=8, auto_approve_save=True)

print(f"\n{'='*60}")
print("Decision log:")
for entry in result["decision_log"]:
    i   = entry.get("iteration", "–")
    t   = entry.get("tool", "reasoning")
    st  = entry.get("result", {}).get("status", "–") if isinstance(entry.get("result"), dict) else "–"
    app = entry.get("approval", "")
    print(f"  [{i}] {t:12s}  status={st:10s}  approval={app}")

<small>**Deliberate failure scenario**  
The search tool may return a timeout, a rate limit response, or an unexpected shape such as a string instead of structured JSON.  
Learners should detect the failure type, retry only when it is temporary, re-plan when the output is malformed, and stop safely when the agent cannot make progress.</small>

<small>This output shows the tool wrapper returning a structured success payload.  
The `source` field tells us whether the result came from the live API or the local fallback, so the downstream agent can handle both cases with the same shape.</small>

In [3]:
# Demonstrate intermittent rate-limit by calling the search tool repeatedly
results = []
for i in range(1, 7):
    res = TOOL_LIBRARY['search'].run(query='agent', top_k=3)
    results.append((i, res))

for i, r in results:
    print(f"Call {i}: {r}")

Call 1: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more than one way, requiring clarification questions from the agent.'}, {'file': 'doc_001.txt', 'score': 1, 'text': 'AI agents are systems that combine reasoning and tool use. They follow the ReAct pattern: alternate thought and action while using tools such as search and email APIs.'}, {'file': 'doc_003.txt', 'score': 1, 'text': 'Error recovery should classify failures, retry intelligently, and re-plan when output is malformed. Agents must log decisions and enforce permission boundaries.'}]}
Call 2: {'status': 'error', 'error_type': 'rate_limit', 'message': 'simulated intermittent rate limit'}
Call 3: {'status': 'success', 'results': [{'file': 'ambiguous_info.txt', 'score': 1, 'text': 'Simulated failure mode: ambiguous information. This entry shows content that could be interpreted in more tha

<small>This trace shows the bounded agent pattern in action: it can search and summarize, but the write step is blocked until approval.  
That demonstrates the safety boundary and the decision log the rubric asks for.</small>

#### Lab Extension: Domain-specific Scenarios

<small>Use the same agent, tools, and corpus to explore how the same ReAct loop behaves across different professional domains.  
Swap the `query` variable in the LLM-driven loop cell above and observe how the tool selection changes.</small>

---

**Finance — Portfolio Risk Research**

```python
query = "What safety boundaries should an AI agent have when handling financial transactions?"
# Corpus file hit: finance_risk.txt, finance_portfolio.txt
# Expected path: search → summarize → save (with approval)
# Key teaching point: the save step is irreversible in a real system → approval gate is non-negotiable
```

---

**Education — Study Planning Assistant**

```python
query = "Summarise how an AI agent can help a student plan their course sequence."
# Corpus file hit: edu_course_planning.txt, edu_study_assistant.txt
# Expected path: search → summarize → save
# Key teaching point: the agent must present options before taking any enrolment action
```

---

**Healthcare — Scheduling Compliance**

```python
query = "What approval and logging requirements apply to agents that schedule medical appointments?"
# Corpus file hit: healthcare_scheduling.txt
# Key teaching point: every write action needs a timestamp, reason, and explicit confirmation
```

---

**E-commerce — Order Tracking with Recovery**

```python
query = "How should an agent handle tool failures when tracking a customer order?"
# Corpus file hit: ecommerce_agent.txt
# Key teaching point: loop limits prevent hammering a temporarily unavailable API
```

---

**Deliberate failure exercise**

```python
# Trigger a timeout and observe recovery
query = "timeout"          # keyword forces SearchTool to return a timeout error
# Expected: agent retries up to MAX_RETRIES, then stops safely

# Trigger malformed output
query = "malformed"        # keyword forces SearchTool to return a plain string instead of JSON
# Expected: agent replans after detecting unexpected_output
```